# Taller-quiz  Tokenizadores de subpalabras

1.  Instalar  los datasets desde   [Hugging Face](https://huggingface.co/) y la libreria de [SentencePiece](https://arxiv.org/pdf/1808.06226)

In [1]:
# Instalación de librerías necesarias
!pip install datasets sentencepiece

In [2]:
from datasets import load_dataset

### Montar Google Drive y definir rutas
Trabajamos en Colab. Montamos Drive para que los archivos de texto y los modelos entrenados **persistan** entre sesiones (si no, se borran al cerrar el entorno).

In [3]:
import os
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Rutas del taller (materia PLN, actividad ACT_1)
BASE_DIR   = '/content/drive/MyDrive/PLN/ACT_1'
DATA_DIR   = os.path.join(BASE_DIR, 'data')      # archivos .txt de entrada
MODELS_DIR = os.path.join(BASE_DIR, 'models')    # modelos .model/.vocab entrenados

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

def model_path(nombre):
    """Devuelve la ruta completa (sin extension) de un modelo dentro de MODELS_DIR."""
    return os.path.join(MODELS_DIR, nombre)

print('BASE_DIR  :', BASE_DIR)
print('DATA_DIR  :', DATA_DIR)
print('MODELS_DIR:', MODELS_DIR)

Mounted at /content/drive
BASE_DIR  : /content/drive/MyDrive/PLN/ACT_1
DATA_DIR  : /content/drive/MyDrive/PLN/ACT_1/data
MODELS_DIR: /content/drive/MyDrive/PLN/ACT_1/models


2. Cargar el dataset de entrenamiento de [word billion ](https://huggingface.co/datasets/jhonparra18/spanish_billion_words_clean) desde  [Hugging Face](https://huggingface.co/).


In [4]:
ds = load_dataset("jhonparra18/spanish_billion_words_clean")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

dataset_infos.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

data/train-00000-of-00018.parquet:   0%|          | 0.00/212M [00:00<?, ?B/s]

data/train-00001-of-00018.parquet:   0%|          | 0.00/229M [00:00<?, ?B/s]

...

data/train-00015-of-00018.parquet:   0%|          | 0.00/210M [00:00<?, ?B/s]

data/train-00016-of-00018.parquet:   0%|          | 0.00/196M [00:00<?, ?B/s]

data/train-00017-of-00018.parquet:   0%|          | 0.00/220M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/46925295 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/18 [00:00<?, ?it/s]

3. Crear una rutina para convertir el dataset en un conjunto de archivos de texto ya que la entrada a [SentencePiece](https://arxiv.org/pdf/1808.06226) son archivos de texto. Es importante tener en cuenta que como  [word billion ](https://huggingface.co/datasets/jhonparra18/spanish_billion_words_clean) es muy grande hay que particionarlo en archivos de texto  para poder entrenarlo con [SentencePiece](https://arxiv.org/pdf/1808.06226).  Aquí un parte del código.




In [5]:
from tqdm.auto import tqdm  # barra de progreso

text_data = []      # buffer de oraciones en memoria
file_count = 0      # contador de archivos generados

# OPCIONAL: el Spanish Billion Words es enorme. Para que el entrenamiento sea
# manejable limitamos la cantidad de archivos. Sube/quita MAX_FILES si quieres
# usar todo el corpus.
MAX_FILES = 20      # 20 archivos * 5000 = 100.000 oraciones aprox.

for sample in tqdm(ds['train']):
    # quitamos saltos de linea: SentencePiece usa '\n' como separador de oraciones
    sample = sample['text'].replace('\n', ' ')
    text_data.append(sample)
    if len(text_data) == 5_000:
        # cada 5000 oraciones volcamos a un archivo dentro de DATA_DIR
        with open(os.path.join(DATA_DIR, f'text_{file_count}.txt'), 'w', encoding='utf-8') as fp:
            fp.write('\n'.join(text_data))
        text_data = []
        file_count += 1
        if file_count >= MAX_FILES:
            break

# guardamos las oraciones sobrantes (el ultimo bloque de menos de 5000)
if text_data:
    with open(os.path.join(DATA_DIR, f'text_{file_count}.txt'), 'w', encoding='utf-8') as fp:
        fp.write('\n'.join(text_data))

print('Archivos generados en', DATA_DIR)

  0%|          | 0/46925295 [00:00<?, ?it/s]

Archivos generados en /content/drive/MyDrive/PLN/ACT_1/data


In [6]:
from pathlib import Path

# SentencePiece recibe la lista de archivos de entrada como una cadena
# separada por comas. Recogemos todos los .txt que generamos en DATA_DIR.
paths = [str(p) for p in Path(DATA_DIR).glob('*.txt')]
input_files = ','.join(paths)

print(f'{len(paths)} archivos de entrada')
print(paths[:3], '...')

20 archivos de entrada
['/content/drive/MyDrive/PLN/ACT_1/data/text_0.txt', '/content/drive/MyDrive/PLN/ACT_1/data/text_1.txt', '/content/drive/MyDrive/PLN/ACT_1/data/text_2.txt'] ...


5. Entrenado el tokenizador  con [SentencePiece](https://arxiv.org/pdf/1808.06226) se deben  usar los siguientes prefijos para tokenizar hasta  cinco sentencias:
- 5.1  Prefijo de bpe con  vocabularios de 10000, 20000 y 30000
- 5.2  Prefijos que usa [WordPiece](https://arxiv.org/abs/2012.15524) tales como <sep> y <cls>
- 5.3  Prefijo para tipo word
- 5.4  Prefijo para tipo char


Primero definimos **cinco sentencias de prueba** que usaremos para comparar cómo tokeniza cada modelo.

In [7]:
import sentencepiece as spm

# Cinco sentencias de prueba (en espanol, acorde al corpus)
sentencias = [
    "El procesamiento de lenguaje natural es una rama de la inteligencia artificial.",
    "SentencePiece tokeniza el texto sin necesidad de un pretokenizador.",
    "Los modelos de subpalabras manejan muy bien las palabras desconocidas.",
    "Buenos Aires es la capital de Argentina y una ciudad muy grande.",
    "Hoy aprenderemos a entrenar un tokenizador con distintos vocabularios.",
]

def tokenizar(prefijo, sentencias=sentencias):
    """Carga un modelo entrenado (.model) y muestra la tokenizacion de cada sentencia.
    `prefijo` es la ruta completa del modelo sin extension (usar model_path('...'))."""
    sp = spm.SentencePieceProcessor(model_file=f'{prefijo}.model')
    print(f'\n===== Modelo: {os.path.basename(prefijo)}  (vocab={sp.get_piece_size()}) =====')
    for s in sentencias:
        piezas = sp.encode(s, out_type=str)   # tokens como texto
        ids = sp.encode(s, out_type=int)      # tokens como ids
        print(f'\nTexto : {s}')
        print(f'Tokens: {piezas}')
        print(f'IDs   : {ids}')
    return sp

### 5.1  Prefijo BPE con vocabularios de 10000, 20000 y 30000
Entrenamos tres modelos **BPE** (`model_type='bpe'`) que solo se diferencian en el tamaño del vocabulario. Comparar los tres permite ver cómo, a mayor vocabulario, las palabras se parten en menos subpalabras.

In [8]:
# 5.1  Entrenamiento de tres modelos BPE
for vocab in [10000, 20000, 30000]:
    spm.SentencePieceTrainer.train(
        input=input_files,
        model_prefix=model_path(f'bpe_{vocab}'),   # se guarda en MODELS_DIR
        vocab_size=vocab,
        model_type='bpe',
        character_coverage=1.0,          # 1.0 para idiomas con alfabeto pequeno (espanol)
        input_sentence_size=200000,      # submuestreo para acelerar el entrenamiento
        shuffle_input_sentence=True,
    )
    print(f'Entrenado bpe_{vocab}')

# Tokenizamos las 5 sentencias con cada vocabulario
for vocab in [10000, 20000, 30000]:
    tokenizar(model_path(f'bpe_{vocab}'))

Entrenado bpe_10000
Entrenado bpe_20000
Entrenado bpe_30000

===== Modelo: bpe_10000  (vocab=10000) =====

Texto : El procesamiento de lenguaje natural es una rama de la inteligencia artificial.
Tokens: ['▁', 'E', 'l', '▁proces', 'amiento', '▁de', '▁lenguaje', '▁natural', '▁es', '▁una', '▁r', 'ama', '▁de', '▁la', '▁inteligencia', '▁artif', 'icial', '.']
IDs   : [9966, 0, 9976, 2534, 490, 5, 6107, 1125, 40, 90, 69, 656, 5, 36, 9858, 5796, 5632, 0]

Texto : SentencePiece tokeniza el texto sin necesidad de un pretokenizador.
Tokens: ['▁', 'S', 'en', 'ten', 'ce', 'P', 'i', 'ece', '▁to', 'ken', 'iza', '▁el', '▁texto', '▁sin', '▁necesidad', '▁de', '▁un', '▁pre', 'to', 'ken', 'iz', 'ador', '.']
IDs   : [9966, 0, 13, 166, 139, 0, 9970, 1817, 88, 8066, 1773, 38, 3034, 238, 3140, 5, 47, 208, 10, 8066, 161, 365, 0]

Texto : Los modelos de subpalabras manejan muy bien las palabras desconocidas.
Tokens: ['▁', 'L', 'os', '▁modelos', '▁de', '▁sub', 'pal', 'ab', 'ras', '▁mane', 'jan', '▁muy', '▁bien',

### 5.2  Prefijos que usa WordPiece: `<sep>` y `<cls>`
Modelos tipo BERT/WordPiece añaden tokens especiales como `<cls>` (inicio) y `<sep>` (separador). En SentencePiece los declaramos con `user_defined_symbols`, de modo que el tokenizador los reconozca como **un único token** y no los parta en subpalabras.

In [9]:
# 5.2  Modelo BPE con los tokens especiales de WordPiece
spm.SentencePieceTrainer.train(
    input=input_files,
    model_prefix=model_path('wordpiece_special'),
    vocab_size=20000,
    model_type='bpe',
    character_coverage=1.0,
    input_sentence_size=200000,
    shuffle_input_sentence=True,
    user_defined_symbols=['<sep>', '<cls>'],   # tokens especiales tipo WordPiece/BERT
)
print('Entrenado wordpiece_special')

# Probamos que <cls> y <sep> se respetan como un solo token
sp_wp = spm.SentencePieceProcessor(model_file=model_path('wordpiece_special') + '.model')
ejemplos_especiales = [f'<cls> {s} <sep>' for s in sentencias]
tokenizar(model_path('wordpiece_special'), ejemplos_especiales)

print('\nIDs de los tokens especiales:')
print('<cls> ->', sp_wp.piece_to_id('<cls>'))
print('<sep> ->', sp_wp.piece_to_id('<sep>'))

Entrenado wordpiece_special

===== Modelo: wordpiece_special  (vocab=20000) =====

Texto : <cls> El procesamiento de lenguaje natural es una rama de la inteligencia artificial. <sep>
Tokens: ['▁', '<cls>', '▁', 'E', 'l', '▁procesamiento', '▁de', '▁lenguaje', '▁natural', '▁es', '▁una', '▁rama', '▁de', '▁la', '▁inteligencia', '▁artificial', '.', '▁', '<sep>']
IDs   : [19966, 4, 19966, 0, 19976, 11560, 7, 6109, 1127, 42, 92, 15576, 7, 38, 9860, 10159, 0, 19966, 3]

Texto : <cls> SentencePiece tokeniza el texto sin necesidad de un pretokenizador. <sep>
Tokens: ['▁', '<cls>', '▁', 'S', 'en', 'ten', 'ce', 'P', 'i', 'ece', '▁to', 'ken', 'iza', '▁el', '▁texto', '▁sin', '▁necesidad', '▁de', '▁un', '▁pre', 'to', 'ken', 'izador', '.', '▁', '<sep>']
IDs   : [19966, 4, 19966, 0, 15, 168, 141, 0, 19970, 1819, 90, 8068, 1775, 40, 3036, 240, 3142, 7, 49, 210, 12, 8068, 11380, 0, 19966, 3]

Texto : <cls> Los modelos de subpalabras manejan muy bien las palabras desconocidas. <sep>
Tokens: ['▁', '<cls>',

### 5.3  Prefijo para tipo `word`
Con `model_type='word'` SentencePiece tokeniza por **palabras completas** (separadas por espacios). No hay subpalabras: cada palabra es un token, y las palabras fuera del vocabulario se marcan como `<unk>`.

In [10]:
# 5.3  Modelo tipo word
spm.SentencePieceTrainer.train(
    input=input_files,
    model_prefix=model_path('word'),
    vocab_size=30000,
    model_type='word',
    character_coverage=1.0,
    input_sentence_size=200000,
    shuffle_input_sentence=True,
)
print('Entrenado word')

tokenizar(model_path('word'))

Entrenado word

===== Modelo: word  (vocab=30000) =====

Texto : El procesamiento de lenguaje natural es una rama de la inteligencia artificial.
Tokens: ['▁El', '▁procesamiento', '▁de', '▁lenguaje', '▁natural', '▁es', '▁una', '▁rama', '▁de', '▁la', '▁inteligencia', '▁artificial.']
IDs   : [0, 5789, 3, 2537, 400, 23, 18, 9730, 3, 6, 4704, 0]

Texto : SentencePiece tokeniza el texto sin necesidad de un pretokenizador.
Tokens: ['▁SentencePiece▁tokeniza', '▁el', '▁texto', '▁sin', '▁necesidad', '▁de', '▁un', '▁pretokenizador.']
IDs   : [0, 7, 982, 43, 1031, 3, 15, 0]

Texto : Los modelos de subpalabras manejan muy bien las palabras desconocidas.
Tokens: ['▁Los', '▁modelos', '▁de', '▁subpalabras', '▁manejan', '▁muy', '▁bien', '▁las', '▁palabras', '▁desconocidas.']
IDs   : [0, 2218, 3, 0, 20504, 39, 69, 14, 347, 0]

Texto : Buenos Aires es la capital de Argentina y una ciudad muy grande.
Tokens: ['▁Buenos▁Aires', '▁es', '▁la', '▁capital', '▁de', '▁Argentina', '▁y', '▁una', '▁ciudad', '▁muy', 

<sentencepiece.SentencePieceProcessor; proxy of <Swig Object of type 'sentencepiece::SentencePieceProcessor *' at 0x7896f8b8a5e0> >

### 5.4  Prefijo para tipo `char`
Con `model_type='char'` el vocabulario son **caracteres individuales**. Es el nivel más fino: nunca hay `<unk>` para texto del mismo idioma, pero las secuencias de tokens son mucho más largas. El `vocab_size` lo determina el conjunto de caracteres, así que aquí no lo fijamos manualmente.

In [11]:
# 5.4  Modelo tipo char
spm.SentencePieceTrainer.train(
    input=input_files,
    model_prefix=model_path('char'),
    model_type='char',
    character_coverage=1.0,
    input_sentence_size=200000,
    shuffle_input_sentence=True,
)
print('Entrenado char')

tokenizar(model_path('char'))

Entrenado char

===== Modelo: char  (vocab=37) =====

Texto : El procesamiento de lenguaje natural es una rama de la inteligencia artificial.
Tokens: ['▁', 'E', 'l', '▁', 'p', 'r', 'o', 'c', 'e', 's', 'a', 'm', 'i', 'e', 'n', 't', 'o', '▁', 'd', 'e', '▁', 'l', 'e', 'n', 'g', 'u', 'a', 'j', 'e', '▁', 'n', 'a', 't', 'u', 'r', 'a', 'l', '▁', 'e', 's', '▁', 'u', 'n', 'a', '▁', 'r', 'a', 'm', 'a', '▁', 'd', 'e', '▁', 'l', 'a', '▁', 'i', 'n', 't', 'e', 'l', 'i', 'g', 'e', 'n', 'c', 'i', 'a', '▁', 'a', 'r', 't', 'i', 'f', 'i', 'c', 'i', 'a', 'l', '.']
IDs   : [3, 0, 13, 3, 17, 10, 6, 14, 4, 8, 5, 16, 7, 4, 9, 12, 6, 3, 11, 4, 3, 13, 4, 9, 18, 15, 5, 27, 4, 3, 9, 5, 12, 15, 10, 5, 13, 3, 4, 8, 3, 15, 9, 5, 3, 10, 5, 16, 5, 3, 11, 4, 3, 13, 5, 3, 7, 9, 12, 4, 13, 7, 18, 4, 9, 14, 7, 5, 3, 5, 10, 12, 7, 24, 7, 14, 7, 5, 13, 0]

Texto : SentencePiece tokeniza el texto sin necesidad de un pretokenizador.
Tokens: ['▁', 'S', 'e', 'n', 't', 'e', 'n', 'c', 'e', 'P', 'i', 'e', 'c', 'e', '▁', 't', 'o', 

<sentencepiece.SentencePieceProcessor; proxy of <Swig Object of type 'sentencepiece::SentencePieceProcessor *' at 0x7896f8d5c960> >

## Conclusiones
- **BPE (5.1):** a mayor `vocab_size` (10k → 20k → 30k), las palabras se parten en **menos** subpalabras (tokens más largos). Es el mejor equilibrio entre tamaño de vocabulario y manejo de palabras raras.
- **WordPiece (5.2):** SentencePiece reproduce el esquema de BERT añadiendo tokens especiales `<cls>`/`<sep>` con `user_defined_symbols`, que se respetan como un único token.
- **word (5.3):** un token por palabra; simple, pero genera muchos `<unk>` con palabras desconocidas.
- **char (5.4):** un token por carácter; sin `<unk>` pero secuencias muy largas.

Esto confirma que con un único framework (SentencePiece) podemos **modelar los demás tokenizadores de subpalabras** variando solo `model_type` y los parámetros de vocabulario.